* **Original paper:** [*Tessa C, Lucetti C, Giannelli M, Diciotti S, Poletti M, Danti S, Baldacci F, Vignali C, Bonuccelli U, Mascalchi M, Toschi N. Progression of brain atrophy in the early stages of Parkinson's disease: a longitudinal tensor-based morphometry study in de novo patients without cognitive impairment. Hum Brain Mapp. 2014 Aug;35(8):3932-44. doi: 10.1002/hbm.22449. Epub 2014 Jan 22. PMID: 24453162; PMCID: PMC6868950.*](https://onlinelibrary.wiley.com/doi/abs/10.1002/hbm.22449)

# Introduction

* **Cohort quote from the paper:** "*Overall, 22 patients (4 women and 18 men, mean age 61.5 ± 8.8) and 17 control subjects (8 women and 9 men, mean age 59.1 ± 8.5 years) completed the study and underwent a second MRI examination. The mean (± standard deviation) follow-up time for patients and controls was 2.8 ± 0.6 (range 2–4) years and 3.9 ± 2.2 (range 2–7) years, respectively. Differences for age between PD patients and control subjects were not significant (P = 0.48, MannWhitney U-test)*"


* Demographics for the PD patients (table taken from the paper):
<img src="./images/original-cohort.png" alt= “” width="30%" height="30%">


* At the baseline evaluation "*No difference in local volume between patients and control subjects was revealed*".


* A (very) brief summary of the main results of the logtitudinal evaluation is:


  * Control subjects: Baseline versus follow-up
    * Control subjects experienced atrophy in several white matter and grey matter regions (Fig. 1a), and cerebrospinal fluid enlargement. There were atrophy clusters involved mainly in white matter, and were more widespread in the frontal lobe.
    
    
  * PD patients: Baseline versus follow-up
    * PD patients showed clusters of reduced white and grey matter volume. These were more evident in the white matter, specially the frontal lobe (Fig. 1b), and showed cerebrospinal fluid enlargement. Grey matter involvement was more widespread than in the control subjects.
    <img src="./images/original-fig1.png" alt= “” width="50%" height="50%">
    
  * PD patients versus control subjects
    * "*PD patients developed bilateral clusters of increased atrophy*" (Fig. 2).
    <img src="./images/original-fig2.png" alt= “” width="50%" height="50%">
    
    
  * Correlation analyses
    * "*In PD patients, no significant correlation between warprates and motor or neuropsychological test scores or their average changes per year between baseline and follow-up were identified*".

# Setup

In [1]:
import numpy as np
import pandas as pd
import os

import livingpark_utils
from livingpark_utils import download
from livingpark_utils import clinical

In [2]:
inputs_dir = os.path.join(os.getcwd(), "inputs/study_files")
outputs_dir = os.path.join(os.getcwd(), "outputs")
data_dir = os.path.join(os.getcwd(), "data")

utils = livingpark_utils.LivingParkUtils()
downloader = download.ppmi.Downloader(utils.study_files_dir)
#random_seed = 1
utils.notebook_init()

This notebook was run on 2023-03-14 07:37:19 UTC +0000


# PPMI cohort preparation

* ### Inclusion/Exclusion criteria I will use to replicate this study:


  * Baseline and ~24 month follow-up T1-weighted MRIs available and usable for TBM. (Magnetic_Resonance_Imaging__MRI_.csv: **MRICMPLT** ??)
  
  
  * Disease duration ~1 year at baseline (PD_Diagnosis_History.csv: **PDDXDT**).
  
  
  * No MCI or dementia (Cognitive_Categorization.csv: **COGSTATE** != 3 for dementia, 2 for MCI)
  
  
  * No depression (Medical_Conditions_Log.csv: **MHCAT**!=115? )
  
  
  * No cardio-vascular autonomic dysfunction (General_Physical_Exam.csv: **PESAQ** == 6 && **ABNORM** == 0)
  
  
  * Subjects: 4 PD women and 18 PD men, Controls: 8 women and 9 men (Primary_Clinical_Diagnosis.csv: **PRIMDIAG==1, 12, 15, 16, 23, 24, 9??? many types of PD.**
  
  
  * Controls: a normal neurological examination (Primary_Clinical_Diagnosis.csv: **PRIMDIAG==17**? Or Neurological_exam.csv battery?), and no relatives with PD (Family_History.csv: ANYFAMPD - *See the notes below on how this differs from the original study*).
  
* ### Some stuff to consider
  * While it's not an inclusion/exclusion criterion, the paper states that  "At follow-up examination, all patients were receiving L-dopa". It makes no mention of L-dopa at baseline.
  * UPDRS, HY, MMSE are not inclusion/exclusion criteria, but where possible will be chosen to be similar to Table. 1 at baseline and follow-up.
  * For the control subjects, the original study states that they must have "No history of familial or personal neurological diseases". I can't find this detailed familial data in PPMI, the existing familial data is only with regard to PD.
  * We are using the PPMI's cognitive state diagnosis for MCI instead of the paper's battery of standardized neuropsychological tests.

In [3]:
# PPMI study data files to reconstruct cohort

required_files = [
    "Demographics.csv",                                   # Sex
    "Age_at_visit.csv",                                   # Age
    "Montreal_Cognitive_Assessment__MoCA_.csv",           # MMSE
    "PD_Diagnosis_History.csv",                           # Disease duration
    "Cognitive_Categorization.csv",                       # MCI diagnosis
    "Participant_Status.csv",                             # Parkinson's vs healthy diagnosis
    "Primary_Clinical_Diagnosis.csv",                     # Subjects with no PD nor other neurological disorder
    "Geriatric_Depression_Scale__Short_Version_.csv",     # GDSS - depression screening
    "Family_History.csv",                                 # PD familial history
    "General_Physical_Exam.csv",                          # Cardio-vascular dysfunction (exclusion)
    "Magnetic_Resonance_Imaging__MRI_.csv",               # Baseline & ~24 month follow-up T1w images
    #"MDS-UPDRS_Part_I.csv",                              # UPDRS (unsure which of all the files I need)
    #"MDS_UPDRS_Part_II__Patient_Questionnaire.csv",      # ""
    "MDS-UPDRS_Part_IV__Motor_Complications.csv",         # ""
    #"MDS-UPDRS_Part_I_Patient_Questionnaire.csv",        # ""
    "MDS-UPDRS_Part_III.csv",                             # Hoehn and Yahr scores
    "Medical_Conditions_Log.csv"                          # to exclude neurological and psychiatric disorders
]

utils.get_study_files(required_files, default=downloader)

Download skipped: No missing files!


In [4]:
#age
age_df = pd.read_csv(os.path.join(inputs_dir, "Age_at_visit.csv"), usecols=["PATNO", "EVENT_ID", "AGE_AT_VISIT"])
print(age_df)

        PATNO EVENT_ID  AGE_AT_VISIT
0        3000       BL          69.1
1        3000      R17          80.5
2        3000       SC          69.1
3        3000      V01          69.4
4        3000      V02          69.6
...       ...      ...           ...
21463  214876       SC          65.4
21464  215976       SC          59.1
21465  216486       SC          57.1
21466  217825       SC          80.9
21467  218338       SC          57.1

[21468 rows x 3 columns]


In [5]:
#sex
sex_df = pd.read_csv(os.path.join(inputs_dir,"Demographics.csv"), usecols=["PATNO", "SEX"])
print(sex_df)

       PATNO  SEX
0       3000    0
1       3001    1
2       3002    0
3       3003    0
4       3004    1
...      ...  ...
2860  218338    1
2861  218340    0
2862  218357    0
2863  218577    0
2864  219311    1

[2865 rows x 2 columns]


In [6]:
#diagnosis
diagnosis_df = pd.read_csv(os.path.join(inputs_dir, "Primary_Clinical_Diagnosis.csv"), usecols=["PATNO", "EVENT_ID", "PRIMDIAG", "OTHNEURO"])
print(diagnosis_df)

        PATNO EVENT_ID  PRIMDIAG               OTHNEURO
0        3000       SC        17                    NaN
1        3000      V04        17                    NaN
2        3000      V06        17                    NaN
3        3000      V08        17                    NaN
4        3000      V10        17                    NaN
...       ...      ...       ...                    ...
14670  206992       BL        25                    NaN
14671  211597       BL        97  REM Behavior disorder
14672  211902       BL         1                    NaN
14673  212243       BL        25                    NaN
14674  212783       BL        25                    NaN

[14675 rows x 4 columns]


In [7]:
#disease duration
disease_duration_df = pd.read_csv(os.path.join(inputs_dir,"PD_Diagnosis_History.csv"), usecols=["PATNO", "EVENT_ID", "PDDXDT"])
print(disease_duration_df)

       PATNO EVENT_ID   PDDXDT
0       3001       SC  04/2010
1       3002       SC  02/2010
2       3003       SC  03/2009
3       3005       SC      NaN
4       3006       SC  11/2010
...      ...      ...      ...
1470  218340       SC  07/2022
1471  218357       SC  02/2023
1472  218577       SC  02/2022
1473  219311       SC  06/2022
1474  219409       SC  01/2023

[1475 rows x 3 columns]


In [8]:
# Cognitive Categorization
cog_cat_df = pd.read_csv(os.path.join(inputs_dir, "Cognitive_Categorization.csv"), usecols=["PATNO", "EVENT_ID", "COGSTATE"])
print(cog_cat_df)

       PATNO EVENT_ID  COGSTATE
0       3000      V08       1.0
1       3000      V10       1.0
2       3000      V12       1.0
3       3000      V14       1.0
4       3000      V15       1.0
...      ...      ...       ...
9833  205186       BL       1.0
9834  206992       BL       1.0
9835  211597       BL       1.0
9836  211902       BL       1.0
9837  212783       BL       1.0

[9838 rows x 3 columns]


In [9]:
#updrs
updrs3_df = pd.read_csv(os.path.join(inputs_dir,"MDS-UPDRS_Part_III.csv"), usecols=["PATNO", "EVENT_ID", "PDSTATE", "PDTRTMNT", "NP3TOT", "NHY"])
print(updrs3_df)

        PATNO EVENT_ID  PDTRTMNT PDSTATE  NP3TOT  NHY
0        3000       BL       NaN     NaN     4.0  0.0
1        3000      V04       NaN     NaN     1.0  0.0
2        3000      V06       NaN     NaN     4.0  0.0
3        3000      V08       NaN     NaN     2.0  0.0
4        3000      V10       NaN     NaN    19.0  0.0
...       ...      ...       ...     ...     ...  ...
20890  206992       BL       0.0     NaN    11.0  0.0
20891  211597       BL       0.0     NaN     3.0  0.0
20892  211902       BL       0.0     NaN    23.0  2.0
20893  212243       BL       0.0     NaN     1.0  0.0
20894  212783       BL       0.0     NaN     3.0  0.0

[20895 rows x 6 columns]


In [10]:
#medical condition
med_cond_df = pd.read_csv(os.path.join(inputs_dir, "Medical_Conditions_Log.csv"), usecols=["PATNO", "EVENT_ID", "MHCAT"]).groupby(['PATNO', 'EVENT_ID'])[['MHCAT']].aggregate(lambda x: tuple(set(x))) # aggregate all codes in a tuple
print(med_cond_df)

                                          MHCAT
PATNO  EVENT_ID                                
3000   ED                            (117, 103)
3001   ED                  (106, 102, 117, 110)
3002   ED                            (114, 117)
3003   ED                            (110, 111)
3004   ED                            (106, 111)
...                                         ...
217825 ED        (102, 104, 106, 110, 112, 115)
217838 ED                            (113, 107)
217865 ED             (106, 110, 115, 116, 117)
219311 ED             (106, 107, 108, 110, 117)
219409 ED                                (106,)

[1787 rows x 1 columns]


In [11]:
#moca --> mmse
moca_df = pd.read_csv(os.path.join(inputs_dir,"Montreal_Cognitive_Assessment__MoCA_.csv"), usecols=["PATNO", "EVENT_ID", "MCATOT"])
moca_df["MMSETOT"] = moca_df["MCATOT"].apply(clinical.moca2mmse)
print(moca_df)

0.0
        PATNO EVENT_ID  MCATOT  MMSETOT
0        3000       SC    27.0     29.0
1        3000      V04    29.0     30.0
2        3000      V06    28.0     30.0
3        3000      V08    30.0     30.0
4        3000      V10    29.0     30.0
...       ...      ...     ...      ...
10022  218338       SC    27.0     29.0
10023  218357       SC    29.0     30.0
10024  218577       SC    29.0     30.0
10025  219311       SC    25.0     28.0
10026  219409       SC    28.0     30.0

[10027 rows x 4 columns]


In [12]:
#parkinsons (COHORT=1) vs healthy control (COHORT=2)
part_stat_df = pd.read_csv(os.path.join(inputs_dir, "Participant_Status.csv"), usecols=["PATNO", "COHORT"])
print(part_stat_df)

       PATNO  COHORT
0       3000       2
1       3001       1
2       3002       1
3       3003       1
4       3004       2
...      ...     ...
2874  218340       1
2875  218357       1
2876  218577       1
2877  219311       1
2878  219409       1

[2879 rows x 2 columns]


In [13]:
#family history
fam_df = pd.read_csv(os.path.join(inputs_dir, "Family_History.csv"), usecols=["PATNO", "EVENT_ID", "ANYFAMPD"])
print(fam_df)

       PATNO EVENT_ID  ANYFAMPD
0       3000       SC       NaN
1       3000    TRANS       0.0
2       3001       SC       NaN
3       3001    TRANS       0.0
4       3001      V15       NaN
...      ...      ...       ...
3778  218340       SC       1.0
3779  218357       SC       0.0
3780  218577       SC       0.0
3781  219311       SC       0.0
3782  219409       SC       0.0

[3783 rows x 3 columns]


In [14]:
#gdss
gdsshort_df = pd.read_csv(os.path.join(inputs_dir,"Geriatric_Depression_Scale__Short_Version_.csv"))
gdsshort_df = gdsshort_df.drop(["REC_ID","PAG_NAME", "INFODT","ORIG_ENTRY","LAST_UPDATE"], axis=1)

# Calculate GDS score for each patient
gds = gdsshort_df.iloc[:, 2:]
gds = gds.agg(['sum'], axis="columns").rename(columns={"sum": "GDSTOT"})

# Add gds score to df
gdsshort_df = pd.concat([gdsshort_df[['PATNO', 'EVENT_ID']], gds], axis=1)
print(gdsshort_df)

        PATNO EVENT_ID  GDSTOT
0        3000       BL     6.0
1        3000      V04     5.0
2        3000      V06     6.0
3        3000      V08     6.0
4        3000      V10     5.0
...       ...      ...     ...
11400  206992       BL     4.0
11401  211597       BL     6.0
11402  211902       BL     3.0
11403  212783       BL     6.0
11404  214876       BL    10.0

[11405 rows x 3 columns]


In [15]:
#physical, PESEQ=6 is cardiovascular

physical_df = pd.read_csv(os.path.join(inputs_dir, "General_Physical_Exam.csv"), usecols=["PATNO", "EVENT_ID", "PESEQ", "PECAT", "ABNORM"])
print(physical_df)

        PATNO EVENT_ID  PESEQ                                  PECAT  ABNORM
0        3000       SC      4                       Ears/Nose/Throat     0.0
1        3000       SC      7                                Abdomen     0.0
2        3000       SC      3                                   Eyes     0.0
3        3000       SC     11  Other (Specify location and describe)     2.0
4        3000       SC      9                           Neurological     0.0
...       ...      ...    ...                                    ...     ...
30489  219409       SC      3                                   Eyes     0.0
30490  219409       SC     11                                  Other     0.0
30491  219409       SC      7                                Abdomen     0.0
30492  219409       SC      4                       Ears/Nose/Throat     0.0
30493  219409       SC      1                                   Skin     0.0

[30494 rows x 5 columns]


In [16]:
#mri availability
mri_df = pd.read_csv(os.path.join(inputs_dir,"Magnetic_Resonance_Imaging__MRI_.csv"), usecols=["PATNO", "EVENT_ID", "MRICMPLT"])
print(mri_df)

       PATNO EVENT_ID  MRICMPLT
0       3000       BL       1.0
1       3000      V12       0.0
2       3001       BL       1.0
3       3002       BL       1.0
4       3003       BL       1.0
...      ...      ...       ...
3530  204559       BL       1.0
3531  206992       BL       1.0
3532  211597       BL       1.0
3533  211902       BL       1.0
3534  212783       BL       1.0

[3535 rows x 3 columns]


# PPMI cohort matching (data aggregation)